# Week 03 - Lesson 02: A2A (Agent-to-Agent)

This notebook demonstrates the implementation of a sophisticated multi-agent system using Google's Agent-to-Agent (A2A) protocol. The A2A protocol represents a breakthrough in agent communication standards, enabling seamless interoperability between AI agents regardless of their underlying frameworks. Through this demonstration, we'll explore how standardized protocols can facilitate complex agent orchestration and collaboration patterns.

<img src="https://storage.googleapis.com/gweb-developer-goog-blog-assets/images/image2.original_6xqVyTd.jpg" alt="A2A illustration" width="1000"/>


### Why Use Google A2A (Agent-to-Agent) Protocol

Google's Agent-to-Agent (A2A) protocol represents a revolutionary approach to AI agent interoperability, establishing a universal communication standard that allows diverse AI agents to seamlessly discover, interact, and collaborate through a unified JSON-RPC infrastructure.  
This groundbreaking framework eliminates the traditional barriers between different AI implementations, creating a truly interoperable ecosystem where agents can work together regardless of their underlying architecture.  

#### 1. Universal Communication Standard

- A2A establishes a uniform JSON-RPC protocol that any AI agent can adopt and implement
    - Agents can establish communication channels without requiring knowledge of each other's internal architecture or implementation details
- The protocol enables real-time streaming capabilities for dynamic, interactive agent conversations

#### 2. Dynamic Agent Discovery and Capability Exposure

- Agents automatically publish their capabilities through standardized metadata structures (AgentCard)
- Each agent declares its specialized skills, supported input/output formats, and operational capabilities
- Host agents can perform real-time discovery of available agent capabilities through the standardized `.well-known/agent-card.json` discovery endpoint

#### 3. Advanced Orchestration and System Composition

- Facilitates the creation of sophisticated multi-agent ecosystems where a central orchestrator can coordinate multiple specialized agents
- Supports both sequential and parallel execution patterns for complex workflow management
- Enables the development of intricate agent collaboration scenarios and automated decision-making pipelines

#### 4. Framework and Infrastructure Agnostic Design

- A2A servers can encapsulate agents developed in any framework, not limited to Google's ADK
- Agents can be deployed as independent microservices across different cloud providers and infrastructure environments
- Promotes architectural flexibility and loose coupling between agent components


---

## What You'll Build

In this tutorial, you'll build an advanced multi-agent intelligence system designed to provide insights into the startup ecosystem. This system demonstrates the power of the A2A protocol by creating three specialized AI agents that work in harmony to deliver comprehensive startup intelligence:

1. **Startup Discovery Scout** - A specialized intelligence agent that scans the market to identify emerging startups across various sectors, focusing on companies with high growth potential and innovative technologies

2. **Startup Benchmarking Analyst** - An analytical agent that performs detailed competitive analysis and benchmarking, providing quantitative insights and market positioning data for discovered startups

3. **Host Agent (Orchestrator)** - A central coordination agent that intelligently manages the workflow between the other agents, ensuring seamless data flow and comprehensive analysis delivery

---

## Environment Setup and Dependencies


In [2]:
# Install required packages
%pip install --upgrade -q a2a-sdk==0.3.0 python-dotenv aiohttp uvicorn requests nest-asyncio
%pip install -q openai litellm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import asyncio
import os
import threading
import time
from typing import Any
from dotenv import load_dotenv

import openai
import httpx
import nest_asyncio
import uvicorn

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

from a2a.types import (AgentCapabilities, AgentCard, AgentSkill, TransportProtocol)
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH

from google.adk.models.lite_llm import LiteLlm
from google.adk.a2a.executor.a2a_agent_executor import (A2aAgentExecutor, A2aAgentExecutorConfig)
from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

In [ ]:
# Set OpenAI API Key
os.environ['OPENAI_API_KEY'] = 'sk-proj-your-openai-api-key'

load_dotenv()

In [4]:
# Function to search the web for current information using OpenAI's web search capabilities
def openai_web_search(query: str) -> str:
    """
    Searches the web for current information using OpenAI's web search capabilities.

    Args:
        query (str): The search query to find current information on the web

    Returns:
        str: Web search results with current information
    """
    try:
        client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        response = client.responses.create(
            model="gpt-4o",
            tools=[{"type": "web_search_preview"}],
            input=f"Search for: {query}. Provide current, accurate information with sources."
        )

        # Extract the text from the response
        search_result = response.output_text if hasattr(response, 'output_text') else str(response)

        return f"Web search results for '{query}':\n{search_result}"

    except Exception as e:
        return f"Error performing web search: {str(e)}"

---

## A2A System

Let's build our three-agent system step by step. We'll create:

1. **Startup Discovery Scout** - Identifies emerging startups across various sectors
2. **Startup Benchmarking Analyst** - Performs competitive analysis and benchmarking
3. **Host Agent** - Orchestrates the other agents (sequentially)

### `1. Agent: Startup Discovery Scout`

This agent searches the web for emerging startups and identifies high-potential companies across various sectors.

In [5]:
# Create the Startup Discovery Scout ADK Agen
openai_model = LiteLlm(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

startup_scout_agent = Agent(
    model=openai_model,
    name='startup_discovery_scout',
    instruction="""
    You are a Senior Startup Intelligence Analyst specializing in discovering emerging startups across various sectors.
    Your expertise lies in identifying high-potential startups for enterprise partnerships, investments, or competitive analysis.

    When asked to find startups:
    1. Use the openai_web_search tool to search for startups in the specified sector/domain
    2. Focus on startups that are:
       - Recently funded (Series A, B, or Seed)
       - Have innovative technology or business models
       - Show strong growth metrics or traction
       - Are relevant to enterprise markets
    3. Extract the top 3 most promising startups
    4. Return them in a structured JSON format

    Search queries should be specific and comprehensive, such as:
    - "fintech startups series A 2024"
    - "AI startups enterprise software recent funding"
    - "healthtech startups series B innovative technology"

    You MUST return your response in the following JSON format:
    {
        "startups": [
            {
                "name": "Startup Name",
                "sector": "Industry/Sector",
                "description": "Brief description of the startup's business model and technology (2-3 sentences)",
                "funding_stage": "Series A/B/Seed",
                "key_innovation": "What makes this startup unique or innovative",
                "enterprise_relevance": "Why this startup is relevant for enterprise clients"
            },
            {
                "name": "Startup Name",
                "sector": "Industry/Sector",
                "description": "Brief description of the startup's business model and technology (2-3 sentences)",
                "funding_stage": "Series A/B/Seed",
                "key_innovation": "What makes this startup unique or innovative",
                "enterprise_relevance": "Why this startup is relevant for enterprise clients"
            },
            {
                "name": "Startup Name",
                "sector": "Industry/Sector",
                "description": "Brief description of the startup's business model and technology (2-3 sentences)",
                "funding_stage": "Series A/B/Seed",
                "key_innovation": "What makes this startup unique or innovative",
                "enterprise_relevance": "Why this startup is relevant for enterprise clients"
            }
        ]
    }

    Only return the JSON object, no additional text.
    """,
    tools=[openai_web_search],
)

print('Startup Discovery Scout Agent created successfully!')

Startup Discovery Scout Agent created successfully!


In [6]:
startup_scout_agent_card = AgentCard(
    name='Startup Discovery Scout',
    url='http://localhost:10020',
    description='Identifies emerging startups across various sectors for enterprise partnerships and investments',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='find_startups',
            name='Find Emerging Startups',
            description='Discovers high-potential startups in specific sectors or domains',
            tags=['startups', 'discovery', 'investment', 'enterprise', 'innovation'],
            examples=[
                "Find fintech startups in Series A",
                'Show me AI startups with enterprise focus',
                'Discover healthtech startups with recent funding',
            ],
        )
    ],
)

In [7]:
remote_startup_scout_agent = RemoteA2aAgent(
    name='find_startups',
    description='Discovers high-potential startups in specific sectors or domains',
    agent_card=f'http://localhost:10020{AGENT_CARD_WELL_KNOWN_PATH}',
)

/tmp/ipython-input-3821966847.py:1: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_startup_scout_agent = RemoteA2aAgent(


### `2. Agent: Startup Benchmarking Analyst`

This agent takes a specific startup and performs competitive analysis with quantitative data and benchmarking.

In [8]:
# Create the Startup Benchmarking Analyst ADK Agent
startup_benchmarking_agent = Agent(
    model=openai_model,
    name='startup_benchmarking_analyst',
    instruction="""
    You are a Senior Competitive Intelligence Analyst specializing in startup benchmarking and market analysis.
    Your expertise lies in comparing startups against established competitors and providing quantitative insights.

    When given a startup to analyze:
    1. Use the openai_web_search tool to search for detailed information about the startup
    2. Search for comparable companies and competitors in the same space
    3. Look for specific metrics and data points:
       - Funding amounts and valuation
       - Revenue and growth rates
       - Market share and customer base
       - Technology differentiation
       - Team and leadership
       - Strategic partnerships
    4. Provide a comprehensive competitive analysis

    Focus on finding concrete numbers, financial data, and measurable metrics.
    Compare the startup against 2-3 established competitors or similar startups.

    You MUST return your response in the following JSON format:
    {
        "analysis": {
            "startup_name": "Startup Name",
            "competitive_landscape": [
                {
                    "competitor": "Competitor/Established Company Name",
                    "comparison_type": "Direct competitor/Similar startup/Established player",
                    "market_position": "Leader/Challenger/Follower",
                    "key_metrics": {
                        "valuation": "Market value or funding amount",
                        "revenue": "Annual revenue if available",
                        "growth_rate": "Growth percentage or trend",
                        "market_share": "Percentage or position in market"
                    },
                    "competitive_advantages": ["Advantage 1", "Advantage 2"],
                    "weaknesses": ["Weakness 1", "Weakness 2"]
                }
            ],
            "startup_strengths": ["Strength 1", "Strength 2", "Strength 3"],
            "startup_weaknesses": ["Weakness 1", "Weakness 2"],
            "market_opportunity": "Assessment of market opportunity and potential",
            "investment_recommendation": "Buy/Watch/Pass with reasoning"
        }
    }

    Only return the JSON object, no additional text.
    """,
    tools=[openai_web_search],
)

print('Startup Benchmarking Analyst Agent created successfully!')

Startup Benchmarking Analyst Agent created successfully!


In [9]:
startup_benchmarking_agent_card = AgentCard(
    name='Startup Benchmarking Analyst',
    url='http://localhost:10021',
    description='Performs competitive analysis and benchmarking of startups with quantitative data',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='analyze_startup',
            name='Analyze Startup',
            description='Provides competitive analysis and benchmarking of a specific startup',
            tags=['analysis', 'benchmarking', 'competitive intelligence', 'startups'],
            examples=[
                'Analyze Stripe competitive position',
                'Benchmark OpenAI against competitors',
                'Provide competitive analysis for Tesla',
            ],
        )
    ],
)

In [10]:
remote_startup_benchmarking_agent = RemoteA2aAgent(
    name='analyze_startup',
    description='Provides competitive analysis and benchmarking of a specific startup',
    agent_card=f'http://localhost:10021{AGENT_CARD_WELL_KNOWN_PATH}',
)

/tmp/ipython-input-521138090.py:1: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_startup_benchmarking_agent = RemoteA2aAgent(


### `3. Agent: Host Agent (Orchestrator)`

The Host Agent coordinates between the other two agents to provide comprehensive startup intelligence analysis.

In [11]:
# Create the Host ADK Agent
host_agent = SequentialAgent(
    name='startup_intelligence_host',
    sub_agents=[remote_startup_scout_agent, remote_startup_benchmarking_agent],
)

In [12]:
host_agent_card = AgentCard(
    name='Startup Intelligence Host',
    url='http://localhost:10022',
    description='Orchestrates, sequentially, startup discovery and competitive analysis using specialized agents',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['application/json'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='comprehensive_startup_analysis',
            name='Comprehensive Startup Analysis',
            description='Finds emerging startups and provides deep competitive analysis of the most promising ones',
            tags=['startups', 'analysis', 'orchestration', 'intelligence', 'competitive'],
            examples=[
                'Find and analyze fintech startups',
                'Discover AI startups and benchmark them',
                'Give me a comprehensive startup intelligence report for healthtech',
            ],
        )
    ],
)

---

## Start Servers

Create function to run each agent as an A2A server and initialize the servers

In [13]:
def create_agent_a2a_server(agent, agent_card):
    """Create an A2A server for any ADK agent.

    Args:
        agent: The ADK agent instance
        agent_card: The ADK agent card

    Returns:
        A2AStarletteApplication instance
    """
    runner = Runner(
        app_name=agent.name,
        agent=agent,
        artifact_service=InMemoryArtifactService(),
        session_service=InMemorySessionService(),
        memory_service=InMemoryMemoryService(),
    )

    config = A2aAgentExecutorConfig()
    executor = A2aAgentExecutor(runner=runner, config=config)

    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )

    # Create A2A application
    return A2AStarletteApplication(
        agent_card=agent_card, http_handler=request_handler
    )

In [14]:
# run_agent_server(), run_servers_in_background() implementation adapted from the official A2A samples repository
# Original source: https://github.com/a2aproject/a2a-samples/blob/main/notebooks/a2a_quickstart.ipynb

# Apply nest_asyncio
nest_asyncio.apply()

# Store server tasks
server_tasks: list[asyncio.Task] = []

# Run the server
async def run_agent_server(agent, agent_card, port) -> None:
    """Run a single agent server."""
    app = create_agent_a2a_server(agent, agent_card)

    config = uvicorn.Config(
        app.build(),
        host='127.0.0.1',
        port=port,
        log_level='warning',
        loop='none',  # Important: let uvicorn use the current loop
    )

    server = uvicorn.Server(config)
    await server.serve()


async def start_all_servers() -> None:
    """Start all servers in the same event loop."""
    # Create tasks for all servers
    tasks = [
        asyncio.create_task(
            run_agent_server(startup_scout_agent, startup_scout_agent_card, 10020)
        ),
        asyncio.create_task(
            run_agent_server(startup_benchmarking_agent, startup_benchmarking_agent_card, 10021)
        ),
        asyncio.create_task(
            run_agent_server(host_agent, host_agent_card, 10022)
        ),
    ]

    # Give servers time to start
    await asyncio.sleep(2)

    print('   - Startup Discovery Scout Agent: http://127.0.0.1:10020')
    print('   - Startup Benchmarking Analyst Agent: http://127.0.0.1:10021')
    print('   - Host Agent: http://127.0.0.1:10022')

    # Keep servers running
    try:
        await asyncio.gather(*tasks)
    except KeyboardInterrupt:
        print('Shutting down servers...')

def run_servers_in_background() -> None:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_all_servers())

# Start the thread
server_thread = threading.Thread(target=run_servers_in_background, daemon=True)
server_thread.start()

# Wait for servers to be ready
time.sleep(3)

/tmp/ipython-input-2220589873.py:19: UserWarning: [EXPERIMENTAL] A2aAgentExecutorConfig: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  config = A2aAgentExecutorConfig()
/tmp/ipython-input-2220589873.py:20: UserWarning: [EXPERIMENTAL] A2aAgentExecutor: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  executor = A2aAgentExecutor(runner=runner, config=config)


   - Startup Discovery Scout Agent: http://127.0.0.1:10020
   - Startup Benchmarking Analyst Agent: http://127.0.0.1:10021
   - Host Agent: http://127.0.0.1:10022


---

## Testing the System

Now let's test our three-agent system. The system works as follows:

1. **Startup Discovery Scout** `Port 10020` - Identifies emerging startups in specific sectors
2. **Startup Benchmarking Analyst** `Port 10021` - Performs competitive analysis and benchmarking
3. **Host Agent** `Port 10022` - Orchestrates both agents to provide comprehensive startup intelligence

In [15]:
# A2ASimpleClient implementation adapted from the official A2A samples repository
# Original source: https://github.com/a2aproject/a2a-samples/blob/main/notebooks/a2a_quickstart.ipynb
class A2ASimpleClient:
    """A2A Simple to call A2A servers."""

    def __init__(self, default_timeout: float = 240.0):
        self._agent_info_cache: dict[
            str, dict[str, Any] | None
        ] = {}  # Cache for agent metadata
        self.default_timeout = default_timeout

    async def create_task(self, agent_url: str, message: str) -> str:
        """Send a message following the official A2A SDK pattern."""
        # Configure httpx client with timeout
        timeout_config = httpx.Timeout(
            timeout=self.default_timeout,
            connect=10.0,
            read=self.default_timeout,
            write=10.0,
            pool=5.0,
        )

        async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
            # Check if we have cached agent card data
            if (
                agent_url in self._agent_info_cache
                and self._agent_info_cache[agent_url] is not None
            ):
                agent_card_data = self._agent_info_cache[agent_url]
            else:
                # Fetch the agent card
                agent_card_response = await httpx_client.get(
                    f'{agent_url}{AGENT_CARD_WELL_KNOWN_PATH}'
                )
                agent_card_data = self._agent_info_cache[agent_url] = (
                    agent_card_response.json()
                )

            # Create AgentCard from data
            agent_card = AgentCard(**agent_card_data)

            # Create A2A client with the agent card
            config = ClientConfig(
                httpx_client=httpx_client,
                supported_transports=[
                    TransportProtocol.jsonrpc,
                    TransportProtocol.http_json,
                ],
                use_client_preference=True,
            )

            factory = ClientFactory(config)
            client = factory.create(agent_card)

            # Create the message object
            message_obj = create_text_message_object(content=message)

            # Send the message and collect responses
            responses = []
            async for response in client.send_message(message_obj):
                responses.append(response)

            # The response is a tuple - get the first element (Task object)
            if (
                responses
                and isinstance(responses[0], tuple)
                and len(responses[0]) > 0
            ):
                task = responses[0][0]  # First element of the tuple

                # Extract text: task.artifacts[0].parts[0].root.text
                try:
                    return task.artifacts[0].parts[0].root.text
                except (AttributeError, IndexError):
                    return str(task)

            return 'No response received'

In [16]:
a2a_client = A2ASimpleClient()

In [17]:
# Testing separetly startup scout agent
async def test_startup_scout() -> None:
    """Test startup scout agent."""
    startup_scout = await a2a_client.create_task(
        'http://localhost:10020', "Search for startups in the AI Agents sector"
    )
    print(startup_scout)

# Run the async function
asyncio.run(test_startup_scout())

/usr/local/lib/python3.12/dist-packages/google/adk/a2a/executor/a2a_agent_executor.py:186: UserWarning: [EXPERIMENTAL] convert_a2a_request_to_adk_run_args: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  run_args = convert_a2a_request_to_adk_run_args(context)
/usr/local/lib/python3.12/dist-packages/google/adk/a2a/converters/request_converter.py:64: UserWarning: [EXPERIMENTAL] convert_a2a_part_to_genai_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcom

```json
{
    "startups": [
        {
            "name": "Mistral",
            "sector": "AI Agents",
            "description": "Mistral is focused on developing large language models and AI agents designed to enhance customer experience through advanced AI-driven interactions.",
            "funding_stage": "Series B",
            "key_innovation": "Specializes in AI agents for customer experience using state-of-the-art LLMs, positioning themselves as leaders in personalized interactions.",
            "enterprise_relevance": "Their products can significantly improve customer service interactions, making them highly relevant for enterprises seeking to enhance their customer support capabilities."
        },
        {
            "name": "Sana",
            "sector": "AI Agents",
            "description": "Sana creates AI agents that integrate seamlessly with enterprise tools to automate tasks and efficiently manage company knowledge.",
            "funding_stage": "Series C",
    

In [18]:
# Testing separetly startup benchmarking agent
async def test_startup_benchmarking() -> None:
    """Test startup benchmarking agent."""
    startup_benchmarking = await a2a_client.create_task(
        'http://localhost:10021', 'Analyze the TinyFish'
    )
    print(startup_benchmarking)


# Run the async function
asyncio.run(test_startup_benchmarking())

```json
{
    "analysis": {
        "startup_name": "TinyFish",
        "competitive_landscape": [
            {
                "competitor": "Adept",
                "comparison_type": "Direct competitor",
                "market_position": "Challenger",
                "key_metrics": {
                    "valuation": "> $400 million in funding",
                    "revenue": "Not publicly available",
                    "growth_rate": "High growth with significant funding support",
                    "market_share": "Growing influence in AI agent market"
                },
                "competitive_advantages": ["Strong financial backing", "Partnerships with Microsoft and Nvidia"],
                "weaknesses": ["Market still developing", "Potential scalability issues"]
            },
            {
                "competitor": "Inflection AI",
                "comparison_type": "Similar startup",
                "market_position": "Challenger",
                "key_metrics": 

In [19]:
# Testing the host agent. This agent will orchestrate the other two agents
async def test_host_analysis() -> None:
    """Test host analysis agent."""
    host_analysis = await a2a_client.create_task(
        'http://localhost:10022',
        'Find startups working in the AI agents for productivity sector and do an analysis of those companies.'
    )
    print(host_analysis)


# Run the async function
asyncio.run(test_host_analysis())

/usr/local/lib/python3.12/dist-packages/google/adk/agents/remote_a2a_agent.py:358: UserWarning: [EXPERIMENTAL] convert_genai_part_to_a2a_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  converted_part = convert_genai_part_to_a2a_part(part)
/usr/local/lib/python3.12/dist-packages/google/adk/a2a/converters/event_converter.py:206: UserWarning: [EXPERIMENTAL] convert_a2a_message_to_event: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  return conver

```json
{
    "analysis": {
        "startup_name": "AI Agents for Productivity Analysis",
        "competitive_landscape": [
            {
                "competitor": "TinyFish",
                "comparison_type": "Direct competitor",
                "market_position": "Challenger",
                "key_metrics": {
                    "valuation": "Approximately $105 million post-Series A (estimated)",
                    "revenue": "Not publicly disclosed, likely in early stages post-Series A",
                    "growth_rate": "Focusing on scaling operations post-Series A funding",
                    "market_share": "Positioned as a notable player in AI-driven enterprise automation"
                },
                "competitive_advantages": [
                    "Innovative automation of web-based tasks",
                    "Early adoption by major clients like Google"
                ],
                "weaknesses": [
                    "Early stages of revenue generation",

---

## Other Resources and References

- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [A2A Protocol Specification](https://github.com/google/a2a)
- [LiteLLM](https://docs.litellm.ai/docs/tutorials/google_adk)
- [ADK Going Multi Model](https://google.github.io/adk-docs/tutorials/agent-team/#step-2-going-multi-model-with-litellm-optional)
- [A2A with Langgraph](https://a2aprotocol.ai/blog/a2a-langraph-tutorial-20250513)
- [A2A Agent Skill and Card](https://a2a-protocol.org/latest/tutorials/python/3-agent-skills-and-card/)
- [OpenAI Web Search Tool](https://platform.openai.com/docs/guides/tools-web-search?api-mode=responses)

---

## Implementation Notes

This tutorial is based on the official A2A samples repository but has been significantly adapted to provide a more accessible learning experience:

**Key Adaptations Made:**

- **Simplified Infrastructure Requirements**: This implementation eliminates the need for Google Cloud Platform (GCP) and Vertex AI accounts, making it more accessible for educational purposes

- **OpenAI Integration**: Replaced Google's search capabilities with OpenAI's web search tool to avoid additional GCP API setup requirements

- **Streamlined Setup**: Removed complex authentication and billing setup steps that could create barriers

**Reference:**
This material is inspired by: [A2A Project Quickstart](https://github.com/a2aproject/a2a-samples/blob/main/notebooks/a2a_quickstart.ipynb)

The core A2A protocol concepts and agent architecture remain consistent with the official implementation, ensuring that students learn the correct patterns while avoiding unnecessary complexity.

---

## Understanding ADK vs A2A: Architectural Decision Guide

*The following comparison content is adpted from the official A2A samples repository to help developers understand when to use each approach:*

### **Using ADK Agents Directly**

```python
# Conceptual Example: Defining Hierarchy
from google.adk.agents import LlmAgent, BaseAgent

# Define individual agents
greeter = LlmAgent(name="Greeter", model="gemini-2.5-pro")
task_doer = BaseAgent(name="TaskExecutor") # Custom non-LLM agent

# Create parent agent and assign children via sub_agents
coordinator = LlmAgent(
    name="Coordinator",
    model="gemini-2.5-pro",
    description="I coordinate greetings and tasks.",
    sub_agents=[ # Assign sub_agents here
        greeter,
        task_doer
    ]
)
```

__Use Direct ADK for Multi-Agents System When:__

- All agents are tightly related and always used together
- Google ADK is the framework choice, and simplicity is prioritized
- Performance of in-process communication is critical
- You don't need distributed deployment
- No built-in service discovery is needed

#### Using ADK Agents Through A2A

__Use A2A for Multi-Agents System When:__

- Building complex multi-agent systems
- Agents need to be developed, deployed, and scaled independently
- You want to integrate agents from different teams or frameworks (Langgraph, CrewAI)
- You need dynamic agent discovery and composition
- Building a platform where agents can be added/removed dynamically
- You want to enable third-party agent integration